In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import torch
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
import pandas as pd

In [2]:

# # Set the base path
# base_path = "/mnt/c/Users/wwwzx/AI_Class_Work/Transformers/"
# print(base_path)


In [ ]:
# Load the saved validation dataset
# training_dataset_path = os.path.join(base_path, "training_dataset.pt")
training = torch.load("training_dataset.pt",weights_only=False)
# print("Loaded traininig  dataset from:", training_dataset_path)
validation=torch.load(val_path,weight_only=False)


In [4]:
# file_path = os.path.join(base_path,"even_further_processed_sales_data_optimized_dtypes.parquet")
data = pd.read_parquet("even_further_processed_sales_data_optimized_dtypes.parquet")


In [5]:
# path_to_check = training_dataset_path  # Replace with the path you want to check
# if os.path.exists(path_to_check):
#     if os.path.isfile(path_to_check):
#         print(f"'{path_to_check}' is a file.")
#     elif os.path.isdir(path_to_check):
#         print(f"'{path_to_check}' is a directory.")
#     else:
#         print(f"'{path_to_check}' exists, but is neither a file nor a directory (e.g., a symbolic link, socket, etc.).")
# else:
#     print(f"'{path_to_check}' does not exist.")

In [ ]:
# validation = TimeSeriesDataSet.from_dataset(
#     training,
#     data,
#     predict=False,
#     stop_randomization=True)

In [ ]:
# Create the validation DataLoader
batch_size = 128
num_workers = 0
val_dataloader = validation.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True if num_workers > 0 else False
)


In [3]:

# Load the model from the checkpoint
best_model = TemporalFusionTransformer.load_from_checkpoint("tft_epoch_epoch=05.ckpt")



C:\Users\WWWZX\anaconda3\envs\ptgp_equv_copy\lib\site-packages\lightning\pytorch\utilities\migration\utils.py:56: The loaded checkpoint was produced with Lightning v2.5.1, which is newer than your current Lightning version: v2.5.0.post0
C:\Users\WWWZX\anaconda3\envs\ptgp_equv_copy\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\WWWZX\anaconda3\envs\ptgp_equv_copy\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [4]:
best_model.eval()


TemporalFusionTransformer(
  	"attention_head_size":               8
  	"categorical_groups":                {}
  	"causal_attention":                  True
  	"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'log_unit_sales', 'group_ids': ['store_nbr', 'item_nbr'], 'weight': None, 'max_encoder_length': 30, 'min_encoder_length': 14, 'min_prediction_idx': 0, 'min_prediction_length': 1, 'max_prediction_length': 1, 'static_categoricals': ['store_nbr', 'item_nbr'], 'static_reals': None, 'time_varying_known_categoricals': ['is_onpromotion', 'is_holiday'], 'time_varying_known_reals': ['day', 'month', 'weekday', 'year', 'transactions'], 'time_varying_unknown_categoricals': [], 'time_varying_unknown_reals': ['log_unit_sales'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': True, 'lags': None, 'add_relative_time_idx': True, 'add_target_scales': True, 'add_encoder_length': True, 'target_normalizer': GroupNormalizer(
  		method='standard

In [8]:
best_model = best_model.cpu()


In [10]:
print("Available hparams:", best_model.hparams)


Available hparams: "attention_head_size":               8
"categorical_groups":                {}
"causal_attention":                  True
"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'log_unit_sales', 'group_ids': ['store_nbr', 'item_nbr'], 'weight': None, 'max_encoder_length': 30, 'min_encoder_length': 14, 'min_prediction_idx': 0, 'min_prediction_length': 1, 'max_prediction_length': 1, 'static_categoricals': ['store_nbr', 'item_nbr'], 'static_reals': None, 'time_varying_known_categoricals': ['is_onpromotion', 'is_holiday'], 'time_varying_known_reals': ['day', 'month', 'weekday', 'year', 'transactions'], 'time_varying_unknown_categoricals': [], 'time_varying_unknown_reals': ['log_unit_sales'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': True, 'lags': None, 'add_relative_time_idx': True, 'add_target_scales': True, 'add_encoder_length': True, 'target_normalizer': GroupNormalizer(
	method='standard',
	groups=['store_nbr'

In [29]:
# Define batch size and lengths
# Extract key parameters from hparams
hparams = best_model.hparams
dataset_params = hparams.dataset_parameters

batch_size = 1
encoder_length = best_model.hparams.dataset_parameters['max_encoder_length']  # 30
decoder_length = best_model.hparams.dataset_parameters['max_prediction_length']  # 1
print(encoder_length)
print(decoder_length)

30
1


In [27]:
sample_input = best_model.example_input_array  # This should give you the expected input format
print(sample_input)

None


In [39]:
# Continuous feature dimensions
continuous_dim = (
    len(hparams.time_varying_reals_encoder) +  # 7 (day, month, weekday, year, transactions, relative_time_idx, log_unit_sales)
    2 +  # add_target_scales (log_unit_sales_center, log_unit_sales_scale)
    1    # add_encoder_length
)  # Total: 10

# Categorical feature dimensions
cat_dim = len(hparams.x_categoricals)  # 4 (store_nbr, item_nbr, is_onpromotion, is_holiday)



In [43]:
# Create dummy input dictionary
dummy_input = {
    "encoder_cont": torch.randn(batch_size, encoder_length, continuous_dim),  # Shape: (1, 30, 10)
    "encoder_cat": torch.zeros((batch_size, encoder_length, cat_dim), dtype=torch.long),  # Shape: (1, 30, 4)
    "decoder_cont": torch.randn(batch_size, decoder_length, continuous_dim),  # Shape: (1, 1, 10)
    "decoder_cat": torch.zeros((batch_size, decoder_length, cat_dim), dtype=torch.long),  # Shape: (1, 1, 4)
    "encoder_lengths": torch.tensor([encoder_length] * batch_size, dtype=torch.long),  # Shape: (1,), value 30
    "decoder_lengths": torch.tensor([decoder_length] * batch_size, dtype=torch.long),  # Shape: (1,), value 1
    "target_scale": torch.tensor([[0.0, 1.0]], dtype=torch.float),  # Shape: (1, 2), mean=0, std=1
}


In [44]:
# Set realistic categorical values based on embedding_sizes
# Order: [store_nbr, item_nbr, is_onpromotion, is_holiday]
cat_ranges = [54, 3995, 2, 2]  # Number of categories for each feature
for i, max_val in enumerate(cat_ranges):
    dummy_input["encoder_cat"][:, :, i] = torch.randint(0, max_val, (batch_size, encoder_length))
    dummy_input["decoder_cat"][:, :, i] = torch.randint(0, max_val, (batch_size, decoder_length))

# Optionally set static categoricals constant across time
dummy_input["encoder_cat"][:, :, 0] = 0  # store_nbr = 0
dummy_input["encoder_cat"][:, :, 1] = 0  # item_nbr = 0
dummy_input["decoder_cat"][:, :, 0] = 0  # store_nbr = 0
dummy_input["decoder_cat"][:, :, 1] = 0  # item_nbr = 0

# Set log_unit_sales placeholder in decoder_cont to 0
dummy_input["decoder_cont"][:, :, -1] = 0  # Assuming last feature is log_unit_sales

In [45]:
# Ensure all tensors are on CPU
for key in dummy_input:
    dummy_input[key] = dummy_input[key].cpu()

# Validate the dummy input with a forward pass
try:
    with torch.no_grad():
        output = best_model(dummy_input)
        print("Forward pass successful!")
        print("Output shape:", output[0].shape)  # Expected: (1, 1, 7) for 7 quantiles, 1 time step
except Exception as e:
    print("Forward pass failed:", e)
    print("Dummy input shapes:")
    for k, v in dummy_input.items():
        print(f"{k}: {v.shape}")
    raise

Forward pass successful!
Output shape: torch.Size([1, 1, 7])


In [46]:
# Define dynamic axes for ONNX export
dynamic_axes = {
    "encoder_cont": {0: "batch_size", 1: "encoder_length"},
    "encoder_cat": {0: "batch_size", 1: "encoder_length"},
    "decoder_cont": {0: "batch_size", 1: "decoder_length"},
    "decoder_cat": {0: "batch_size", 1: "decoder_length"},
    "encoder_lengths": {0: "batch_size"},
    "decoder_lengths": {0: "batch_size"},
    "target_scale": {0: "batch_size"},
}


In [49]:
# Trace the model with TorchScript, bypassing trainer check
try:
    # Temporarily set _jit_is_scripting to True to skip trainer check
    original_jit_flag = getattr(best_model, '_jit_is_scripting', False)
    best_model._jit_is_scripting = True
    traced_model = torch.jit.trace(best_model, (dummy_input,), strict=False)
    best_model._jit_is_scripting = original_jit_flag  # Reset the flag
    print("Model tracing successful!")
except Exception as e:
    best_model._jit_is_scripting = original_jit_flag  # Ensure flag is reset on failure
    print("Model tracing failed:", e)
    raise


C:\Users\WWWZX\anaconda3\envs\ptgp_equv_copy\lib\site-packages\pytorch_forecasting\models\temporal_fusion_transformer\_tft.py:520: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  max_encoder_length = int(encoder_lengths.max())
C:\Users\WWWZX\anaconda3\envs\ptgp_equv_copy\lib\site-packages\pytorch_forecasting\models\nn\rnn.py:104: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert min_length >= 0, "sequence lengths must be great equals 0"
C:\Users\WWWZX\anaconda3\envs\ptgp_equv_copy\lib\site-packages\pytorch_forecasting\models\nn\rnn.py:106: TracerWarni

Model tracing successful!


In [55]:
# Save the traced model to a file
traced_model_path = "tft_traced_model.pt"
try:
    traced_model.save(traced_model_path)
    print(f"Traced model saved successfully to {traced_model_path}!")
except Exception as e:
    print("Failed to save traced model:", e)
    raise

# Optional: Verify the saved model by loading and running it
try:
    loaded_model = torch.jit.load(traced_model_path)
    with torch.no_grad():
        loaded_output = loaded_model(dummy_input)
        print("Loaded model forward pass successful!")
        print("Loaded model output shape:", loaded_output[0].shape)  # Expected: (1, 1, 7)
except Exception as e:
    print("Failed to load or run traced model:", e)

Traced model saved successfully to tft_traced_model.pt!
Loaded model forward pass successful!
Loaded model output shape: torch.Size([1, 1, 7])


In [56]:
import torch

# Load the traced model
loaded_model = torch.jit.load("tft_traced_model.pt")

# Prepare input data (must match the structure used during tracing)
input_data = {
    "encoder_cont": torch.randn(1, 30, 10, dtype=torch.float32),  # 10 continuous features, 30 time steps
    "encoder_cat": torch.zeros((1, 30, 4), dtype=torch.long),     # 4 categorical features
    "decoder_cont": torch.randn(1, 1, 10, dtype=torch.float32),   # 10 continuous features, 1 time step
    "decoder_cat": torch.zeros((1, 1, 4), dtype=torch.long),      # 4 categorical features
    "encoder_lengths": torch.tensor([30], dtype=torch.long),      # Length of encoder sequence
    "decoder_lengths": torch.tensor([1], dtype=torch.long),       # Length of decoder sequence
    "target_scale": torch.tensor([[0.0, 1.0]], dtype=torch.float32),  # Mean=0, std=1 for denormalization
}

# Set static categoricals and placeholder values (as during tracing)
input_data["encoder_cat"][:, :, 0] = 0  # store_nbr
input_data["encoder_cat"][:, :, 1] = 0  # item_nbr
input_data["decoder_cat"][:, :, 0] = 0  # store_nbr
input_data["decoder_cat"][:, :, 1] = 0  # item_nbr
input_data["decoder_cont"][:, :, -1] = 0  # log_unit_sales placeholder

# Run inference
with torch.no_grad():
    output = loaded_model(input_data)
    print("Inference output shape:", output[0].shape)  # [1, 1, 7] for 7 quantiles
    print("Predictions:", output[0])  # Predicted quantiles for log_unit_sales

Inference output shape: torch.Size([1, 1, 7])
Predictions: tensor([[[0.1305, 0.2569, 0.5805, 0.9764, 1.4414, 1.8813, 2.4702]]])


In [ ]:
# Debug: Check what predict returns
# Make predictions with mode="raw" and return_x=True
result = best_model.predict(val_dataloader, mode="raw", return_x=True)
print("Result type:", type(result))
# print("Result:", result)




In [ ]:
raw_predictions = result.output  # The raw output dictionary


In [ ]:
subset_size = 100
subset_data = list(iter(val_dataloader))[:subset_size]  # Take first 100 batches
    



In [ ]:
# Create a new DataLoader from the subset
subset_loader = DataLoader(subset_data, batch_size=val_dataloader.batch_size, shuffle=False)



In [ ]:

# Run prediction
raw_predictions = best_model.predict(subset_loader, mode="raw", return_x=True)

In [ ]:
encoder_lengths = raw_predictions["encoder_lengths"]
print(f"Encoder lengths in raw_predictions: {encoder_lengths}")
print(f"Max encoder length in raw_predictions: {encoder_lengths.max().item()}")



###### for batch in val_dataloader:
    x, y = batch
    encoder_lengths = x["encoder_lengths"]
    print(f"Encoder lengths: {encoder_lengths}")
    print(f"Max encoder length: {encoder_lengths.max().item()}")
    break



In [ ]:
# # Truncate encoder lengths and attention weights in raw_predictions
# max_encoder_length = best_model.hparams.max_encoder_length  # Should be 30
# print(max_encoder_length)
# for idx in range(len(raw_predictions["encoder_lengths"])):
#     encoder_length = raw_predictions["encoder_lengths"][idx].item()
#     if encoder_length > max_encoder_length:
#         print(f"Truncating encoder_length from {encoder_length} to {max_encoder_length} at index {idx}")
#         # Truncate the encoder length
#         raw_predictions["encoder_lengths"][idx] = max_encoder_length
#         # Truncate the encoder attention weights
#         raw_predictions["encoder_attention"][idx] = raw_predictions["encoder_attention"][idx][..., :max_encoder_length]



In [ ]:
# Interpret the model predictions
# Now interpret the modified raw_predictions
interpretation = best_model.interpret_output(raw_predictions, reduction="sum")

In [ ]:
# Plot the interpretation
best_model.plot_interpretation(interpretation)

In [ ]:
print(f"Model's max_encoder_length: {best_model.hparams.max_encoder_length}")

In [ ]:
for batch in val_dataloader:
    x, y = batch
    encoder_lengths = x["encoder_lengths"]
    if encoder_lengths.max().item() > 30:  # Hardcoded limit
        print(f" Found encoder length > 30: {encoder_lengths.max().item()}")
        break  # Stop early

In [ ]:
print(f"Model max_encoder_length: {best_model.hparams.max_encoder_length}")

In [ ]:
print(validation.__dict__.keys())  # Check available attributes

In [ ]:
print(validation.get_parameters())  # See dataset parameters

In [ ]:
for batch in val_dataloader:
    x, y = batch
    if "encoder_lengths" in x:
        print(f"Max encoder length in validation batch: {x['encoder_lengths'].max().item()}")
    else:
        print(" 'encoder_lengths' not found in batch!")
    break


In [ ]:
# Extract the raw prediction outputs
q10 = raw_predictions.output[..., 0]  # 10% quantile
q50 = raw_predictions.output[..., 1]  # 50% quantile (Median)
q90 = raw_predictions.output[..., 2]  # 90% quantile

# Approximate Q1 (25%) and Q3 (75%) using interpolation
q25 = 0.5 * (q10 + q50)  # Linear interpolation
q75 = 0.5 * (q50 + q90)  # Linear interpolation

# Print the quartile values
print(f"Q1 (25%): {q25}")
print(f"Q2 (Median, 50%): {q50}")
print(f"Q3 (75%): {q75}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(q25, label="Q1 (25%)")
plt.plot(q50, label="Q2 (Median)")
plt.plot(q75, label="Q3 (75%)")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Predicted Value")
plt.title("TFT Predicted Quartiles")
plt.show()


In [ ]:
print(f"Available keys in raw_predictions: {quantile_predictions.keys()}")


In [ ]:
print(best_model.hparams)
print(f"Model's max_encoder_length: {best_model.hparams.max_encoder_length}")
